# Black Summer Liability — End-to-End Attribution

**Event**: 2019–20 Australian bushfire season (Black Summer)  
**Dates**: October 2019 – March 2020  
**Region**: Southeastern Australia

## Methodology

This notebook implements the full attribution chain for a single event:

```
Entity warming share  ×  FAR  ×  Total damages  =  Entity liability estimate
```

Where:
- **Entity warming share** = entity's warming contribution / total anthropogenic warming (from FaIR, `02-attribution/01`)
- **FAR** = 1 − 1/PR, derived from the WWA attribution study (van Oldenborgh et al. 2021)
- **Total damages** = several scenarios from published estimates

## Key sources
- Attribution: van Oldenborgh et al. (2021) *Nat. Hazards Earth Syst. Sci.* https://doi.org/10.5194/nhess-21-941-2021
- Damages: Insurance Council of Australia; Filkov et al. (2020); Parliamentary Budget Office (2020)
- Entity warming: `data/processed/entity_warming_contribution.parquet`

## Important caveats
1. Entity warming shares are **global** — the link to Australian regional fire weather is via the proportionality assumption (global ΔT drives regional FWI proportionally). This is an approximation.
2. PR values from WWA compare current climate to ~1900, not a pure counterfactual — they are conservative lower bounds.
3. Damage estimates vary widely; we carry three scenarios throughout.
4. **Physical attribution ≠ legal liability.** These are risk-proportional estimates, not legal determinations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

PROC = Path("../../data/processed")
FIGS = Path("../../outputs/figures")

## 1. Event parameters

### Probability Ratio (PR) — from WWA study

Van Oldenborgh et al. (2021) report two PR estimates for southeastern Australian fire weather relative to ~1900 climate:

| Metric | PR | FAR | Notes |
|--------|-----|-----|-------|
| Fire Weather Index (FWI) | ≥4 | ≥0.75 | Models underestimate; lower bound |
| Monthly Severity Rating (MSR) | ≥9 | ≥0.89 | Higher-end estimate |
| Heat component alone | ~10 | ~0.90 | Models underestimate observed trend |

We use **PR=4 as the conservative case** and **PR=9 as the central case**, and propagate both through the liability calculation.

### Damage estimates

Three scenarios reflecting different damage accounting approaches:

| Scenario | AUD (B) | USD (B) | Source | Notes |
|----------|---------|---------|--------|-------|
| Conservative (insured) | 2.32 | ~1.6 | Insurance Council of Australia | Severely underestimates — most bush properties uninsured |
| Central (direct economic) | 10.0 | ~6.9 | Parliamentary Budget Office, sectoral studies | Property + agriculture + tourism + health |
| Comprehensive (total social) | 103.0 | ~71.1 | Filkov et al. 2020; Deloitte Access Economics | Includes ecosystem, mental health, long-run productivity |

In [ ]:
AUD_TO_USD = 0.69  # approximate 2020 average exchange rate

# PR scenarios — (label, pr_low, pr_central, pr_high)
# We treat the WWA FWI value as conservative low and MSR as central
# Upper bound uses heat component and explicit model-underestimation caveat
PR_LOW     = 4.0
PR_CENTRAL = 9.0
PR_HIGH    = 15.0   # plausible upper bound given model underestimation

def far(pr):
    """Fraction of Attributable Risk from Probability Ratio."""
    return 1.0 - 1.0 / pr

far_low     = far(PR_LOW)
far_central = far(PR_CENTRAL)
far_high    = far(PR_HIGH)

print("FAR by PR scenario:")
print(f"  Conservative (PR={PR_LOW:.0f}):  FAR = {far_low:.3f}  ({far_low*100:.1f}% of damages attributable to climate change)")
print(f"  Central      (PR={PR_CENTRAL:.0f}):  FAR = {far_central:.3f}  ({far_central*100:.1f}%)")
print(f"  Upper bound  (PR={PR_HIGH:.0f}): FAR = {far_high:.3f}  ({far_high*100:.1f}%)")

# Damage scenarios in USD billions
damages = {
    "Conservative\n(insured, AUD 2.3B)": 2.32 * AUD_TO_USD,
    "Central\n(direct economic, AUD 10B)": 10.0 * AUD_TO_USD,
    "Comprehensive\n(total social, AUD 103B)": 103.0 * AUD_TO_USD,
}

print("\nDamage scenarios (USD billions):")
for label, usd in damages.items():
    label_short = label.split("\n")[0]
    print(f"  {label_short:<15}  USD {usd:.2f}B")

## 2. Entity warming shares

Load per-entity warming contributions from the FaIR analysis. Each entity's **share of total anthropogenic warming** is their proportional contribution to the forcing that raised Black Summer fire risk.

In [ ]:
ew = pd.read_parquet(PROC / "entity_warming_contribution.parquet")

# Total anthropogenic warming from FaIR median (p50)
total_warming_p50 = ew["warming_p50_degC"].sum()

# Entity share of total Carbon Majors warming
# Note: Carbon Majors covers ~45% of global fossil CO2; we attribute only that fraction
ew["cm_warming_share"] = ew["warming_p50_degC"] / total_warming_p50
ew["cm_warming_share_p05"] = ew["warming_p05_degC"] / ew["warming_p05_degC"].sum()
ew["cm_warming_share_p95"] = ew["warming_p95_degC"] / ew["warming_p95_degC"].sum()

# Total warming covered by Carbon Majors entities
total_p50 = ew["warming_p50_degC"].sum()
total_p05 = ew["warming_p05_degC"].sum()
total_p95 = ew["warming_p95_degC"].sum()

print(f"Carbon Majors total attributed warming (p50): {total_p50*1000:.1f} m°C = {total_p50:.4f} °C")
print(f"  5th–95th percentile: [{total_p05*1000:.1f}, {total_p95*1000:.1f}] m°C")
print(f"\nTop 10 entities by warming share:")
top10 = ew.nlargest(10, "warming_p50_degC")[["parent_entity", "parent_type", "warming_p50_degC", "cm_warming_share"]]
top10["warming_m_degC"] = top10["warming_p50_degC"] * 1000
top10["share_pct"] = top10["cm_warming_share"] * 100
print(top10[["parent_entity", "parent_type", "warming_m_degC", "share_pct"]].to_string(index=False))

## 3. Liability calculation

For each entity:

```
liability_USD = damages_USD × FAR × entity_cm_warming_share
```

We compute across all combinations of damage scenario × PR scenario to produce a full uncertainty matrix.

In [ ]:
LIABILITY_SCENARIOS = {
    "conservative": {"damages_usd_b": 2.32 * AUD_TO_USD, "far": far_low,     "pr": PR_LOW},
    "central":      {"damages_usd_b": 10.0 * AUD_TO_USD, "far": far_central, "pr": PR_CENTRAL},
    "comprehensive":{"damages_usd_b": 103.0 * AUD_TO_USD, "far": far_high,   "pr": PR_HIGH},
}

liability = ew[["parent_entity", "parent_type", "cm_warming_share",
                "cm_warming_share_p05", "cm_warming_share_p95",
                "warming_p50_degC"]].copy()

for scenario, params in LIABILITY_SCENARIOS.items():
    d = params["damages_usd_b"]
    f = params["far"]
    liability[f"liability_{scenario}_USD_M"] = (
        liability["cm_warming_share"] * f * d * 1000  # convert B to M
    )

# Uncertainty range on the central scenario using FaIR p05/p95
d_central = LIABILITY_SCENARIOS["central"]["damages_usd_b"]
f_central = LIABILITY_SCENARIOS["central"]["far"]
liability["liability_central_p05_USD_M"] = liability["cm_warming_share_p05"] * f_central * d_central * 1000
liability["liability_central_p95_USD_M"] = liability["cm_warming_share_p95"] * f_central * d_central * 1000

liability = liability.sort_values("liability_central_USD_M", ascending=False).reset_index(drop=True)
liability["rank"] = liability.index + 1

print(f"Central scenario: damages = USD {d_central:.2f}B, FAR = {f_central:.3f} (PR={PR_CENTRAL:.0f})")
print(f"Total attributed damages (Carbon Majors share): USD {liability['liability_central_USD_M'].sum()/1000:.2f}B")
print()
print("Top 15 entities — central liability estimate (USD millions):")
top15_display = liability.head(15)[[
    "rank", "parent_entity", "parent_type",
    "liability_conservative_USD_M", "liability_central_USD_M", "liability_comprehensive_USD_M"
]].copy()
top15_display.columns = ["Rank", "Entity", "Type", "Conservative $M", "Central $M", "Comprehensive $M"]
for col in ["Conservative $M", "Central $M", "Comprehensive $M"]:
    top15_display[col] = top15_display[col].map("{:,.1f}".format)
print(top15_display.to_string(index=False))

## 4. Visualisation — liability by entity and scenario

In [ ]:
top20 = liability.head(20).copy()

type_colors = {
    "Investor-owned Company": "#2196F3",
    "State-owned Entity":     "#FF5722",
    "Nation State":           "#4CAF50",
}
colors = top20["parent_type"].map(type_colors)

fig, ax = plt.subplots(figsize=(11, 8))

y = np.arange(len(top20))
ax.barh(y, top20["liability_comprehensive_USD_M"], color=colors, alpha=0.25, label="Comprehensive")
ax.barh(y, top20["liability_central_USD_M"], color=colors, alpha=0.7, label="Central")
ax.barh(y, top20["liability_conservative_USD_M"], color=colors, alpha=1.0, label="Conservative")

# Uncertainty bars (FaIR p05–p95) on central estimate — clip to zero to avoid negative deltas
xerr_lo = np.maximum(0, top20["liability_central_USD_M"] - top20["liability_central_p05_USD_M"])
xerr_hi = np.maximum(0, top20["liability_central_p95_USD_M"] - top20["liability_central_USD_M"])
ax.errorbar(
    top20["liability_central_USD_M"], y,
    xerr=[xerr_lo, xerr_hi],
    fmt="none", color="#333", linewidth=1, capsize=3, alpha=0.6,
    label="Climate model uncertainty (5–95th)",
)

ax.set_yticks(y)
ax.set_yticklabels(top20["parent_entity"], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Attributed liability (USD millions)")
ax.set_title(
    "Black Summer 2019–20: attributed liability by entity\n"
    "(proportional to warming contribution × FAR × damages)",
    fontsize=12,
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))

from matplotlib.patches import Patch
entity_legend = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
scenario_legend = [
    Patch(facecolor="grey", alpha=1.0, label="Conservative (insured, PR=4)"),
    Patch(facecolor="grey", alpha=0.6, label="Central (direct, PR=9)"),
    Patch(facecolor="grey", alpha=0.2, label="Comprehensive (total social, PR=15)"),
]
ax.legend(handles=entity_legend + scenario_legend, fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIGS / "black_summer_liability_top20.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scenario comparison — total Carbon Majors attributed liability
scenario_totals = pd.DataFrame([
    {
        "scenario": name,
        "damages_usd_b": p["damages_usd_b"],
        "far": p["far"],
        "pr": p["pr"],
        "total_attributed_usd_b": liability[f"liability_{name}_USD_M"].sum() / 1000,
    }
    for name, p in LIABILITY_SCENARIOS.items()
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: waterfall of damage → FAR-adjusted → CM-share
ax = axes[0]
for i, row in scenario_totals.iterrows():
    bars = ax.bar(
        [i*3, i*3+1, i*3+2],
        [row.damages_usd_b, row.damages_usd_b * row.far, row.total_attributed_usd_b],
        color=["#90A4AE", "#FF7043", "#42A5F5"], alpha=0.85
    )
ax.set_xticks([1, 4, 7])
ax.set_xticklabels([s.title() for s in LIABILITY_SCENARIOS.keys()], fontsize=9)
ax.set_ylabel("USD billions")
ax.set_title("Damage attribution funnel by scenario", fontsize=11)
ax.legend(
    [plt.Rectangle((0,0),1,1, color=c, alpha=0.85) for c in ["#90A4AE", "#FF7043", "#42A5F5"]],
    ["Total damages", "Climate-attributed (×FAR)", "Carbon Majors share"],
    fontsize=8
)

# Right: entity type breakdown of central liability
ax2 = axes[1]
type_breakdown = (
    liability.groupby("parent_type")["liability_central_USD_M"]
    .sum()
    .sort_values(ascending=False)
)
type_breakdown.plot.bar(ax=ax2, color=[type_colors[t] for t in type_breakdown.index], rot=20)
ax2.set_title("Central scenario: liability by entity type", fontsize=11)
ax2.set_ylabel("USD millions")
ax2.set_xlabel("")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
plt.tight_layout()
plt.savefig(FIGS / "black_summer_scenario_comparison.png", bbox_inches="tight")
plt.show()

print("\nScenario summary:")
print(scenario_totals.to_string(index=False))

## 5. Sensitivity analysis

How does the top entity's (Saudi Aramco's) attributed liability vary across the full PR × damage matrix?

In [ ]:
aramco_share = liability.loc[liability["parent_entity"] == "Saudi Aramco", "cm_warming_share"].values[0]

pr_range     = [2, 3, 4, 6, 9, 12, 15, 20]
damage_range = [2.32*AUD_TO_USD, 5*AUD_TO_USD, 10*AUD_TO_USD, 25*AUD_TO_USD, 50*AUD_TO_USD, 103*AUD_TO_USD]

grid = pd.DataFrame(
    index=[f"PR={p}" for p in pr_range],
    columns=[f"AUD {d/AUD_TO_USD:.0f}B" for d in damage_range],
    data=[
        [aramco_share * far(p) * d * 1000 for d in damage_range]
        for p in pr_range
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    grid.astype(float), ax=ax,
    fmt=".0f", annot=True, cmap="YlOrRd",
    cbar_kws={"label": "USD millions"},
    linewidths=0.5
)
ax.set_title("Saudi Aramco — Black Summer liability sensitivity (USD millions)\nby Probability Ratio × damage estimate", fontsize=11)
ax.set_xlabel("Total damages (AUD)")
ax.set_ylabel("Probability Ratio (PR)")
plt.tight_layout()
plt.savefig(FIGS / "black_summer_sensitivity_aramco.png", bbox_inches="tight")
plt.show()

print(f"\nAramco warming share: {aramco_share*100:.2f}% of Carbon Majors total")

## 6. Save outputs

In [ ]:
liability.to_parquet(PROC / "black_summer_liability.parquet", index=False)
scenario_totals.to_csv(PROC / "black_summer_scenario_totals.csv", index=False)

print("Saved:")
print(f"  black_summer_liability.parquet       — {len(liability)} rows (one per entity)")
print(f"  black_summer_scenario_totals.csv     — scenario summary")
print()
print("Key results:")
for name, params in LIABILITY_SCENARIOS.items():
    total = liability[f"liability_{name}_USD_M"].sum() / 1000
    top1  = liability.iloc[0]
    print(f"  {name.title():<15}  total CM liability = USD {total:.2f}B")
    print(f"                    top entity = {top1.parent_entity} (USD {top1[f'liability_{name}_USD_M']:.1f}M)")

## Key findings

Update after running.

- **Total Carbon Majors liability (central scenario)**: USD ___B
- **Top entity**: ___ (USD ___M central)
- **FAR range**: 75–93% of damages attributable to climate change across PR scenarios
- **Biggest uncertainty driver**: damage estimate (50× range from insured to total social cost), not the attribution science

→ See `wiki/findings/2026-05-15-black-summer-liability.md`